## Imports

In [0]:
pip install pyyaml

In [0]:
%run ./tiresias_utils

In [0]:
import codecs
import xml.etree.ElementTree as ET
from pyspark.sql.types import StructType, StructField, StringType
from pyspark.sql.functions import expr
import zipfile
import os
import yaml
from pyspark.sql import functions as F
import pandas as pd
from pyspark.sql.functions import col, lit, expr, when, current_timestamp, from_utc_timestamp
import sys
sys.path.append("../aws_utils")
import aws_utils

## Parameters & Paths

In [0]:
env = aws_utils.get_env_prefix()

In [0]:
# Ensure required volumes exist before proceeding
spark.sql(f"CREATE VOLUME IF NOT EXISTS `{env}-cdh-delivery`.tiresias.rejection_files")
spark.sql(f"CREATE VOLUME IF NOT EXISTS `{env}-cdh-delivery`.tiresias.exports")

In [0]:
spark.sql(f"""
    CREATE TABLE IF NOT EXISTS `{env}-cdh-delivery`.tiresias.old_style_rejections (
        id BIGINT GENERATED ALWAYS AS IDENTITY,
        export_type STRING,
        export_date DATE,
        IngestionTimestamp timestamp,
        rejection_file STRING,
        vendor_code INT,
        account_number STRING,
        Customer_1 STRING,
        Customer_2 STRING,
        error_code STRING,
        created TIMESTAMP,
        cl_sk_id INT
    )
    USING DELTA;
""")

In [0]:
main_path = f"/Volumes/{env}-cdh-delivery/tiresias"
print(main_path)

config = load_config()

# Extract file paths
format_files_path = config["file_paths"]["format_files_path"]
extracted_files_path = config["file_paths"]["extracted_files_path"]
zip_files_path = config["file_paths"]["zip_files_path"]

# Print to verify
print(f"Format Files Path: {format_files_path}")
print(f"Extracted Files Path: {extracted_files_path}")
print(f"Zip Files Path: {zip_files_path}")

## Date

In [0]:
export_date_df = spark.sql("""
    SELECT last_day(add_months(cast(getdate() as date), -2)) AS export_date
""")

export_date = export_date_df.collect()[0]['export_date']
# Format the date to match the "MM.YYYY" in the zip file name-->for check
# For 2025-09-30, this will create the string "09.2025"
expected_date_str = export_date.strftime("%m.%Y")

print(f"Calculated export_date: {export_date}")
print(f"Expecting zip files containing: {expected_date_str}")

## Deleta Data

In [0]:
target_table = f"`{env}-cdh-delivery`.tiresias.old_style_rejections"

has_data_for_date = spark.sql(f"""
    SELECT 1
    FROM {target_table}
    WHERE export_date = '{export_date}'
    LIMIT 1
""").count() > 0

if has_data_for_date:
    print(f"Existing data found for export_date={export_date}. Deleting before reload.")

    spark.sql(f"""
        DELETE FROM {target_table}
        WHERE export_date = '{export_date}'
    """)
else:
    print(f"No existing data found for export_date={export_date}. Proceeding.")

## Process Of Zip Files

In [0]:
# Get the full zip path 
full_zip_path = f"{main_path}/{zip_files_path}"

print(f"Validation: Expecting files with: {expected_date_str}")
print(f"Checking directory: {full_zip_path}")

# 1. Scan the directory
try:
    all_zips = dbutils.fs.ls(full_zip_path)
except Exception as e:
    if "java.io.FileNotFoundException" in str(e):
        print("No rejections files were loaded due to empty directory.")
        dbutils.notebook.exit("Skipping: Input directory not found.")
    else:
        raise e

matching_files = []
unexpected_files = []

for zip_file in all_zips:
    if zip_file.name.endswith(".zip"):
        if expected_date_str in zip_file.name:
            matching_files.append(zip_file.name)
        else:
            unexpected_files.append(zip_file.name)

# 2. Decide to Continue or Skip
if unexpected_files:
    msg = f"No rejections files were loaded due to wrong date. Found unexpected files: {unexpected_files}"
    print(msg)
    dbutils.notebook.exit(f"Skipping: {msg}")
elif not matching_files:
    msg = f"No rejections files were loaded due to empty directory or no matching files found for {expected_date_str}."
    print(msg)
    dbutils.notebook.exit(f"Skipping: {msg}")
else:
    print(f" Validation passed: Found {len(matching_files)} matching file(s): {matching_files}")
    print("Continuing to unzip...")

In [0]:
# Define file paths
full_extracted_path = f"{main_path}/{extracted_files_path}"
print(full_extracted_path)

unzip_files(full_zip_path, full_extracted_path)

In [0]:
files_list = dbutils.fs.ls(full_extracted_path)
files_list_txt = [file for file in files_list if not file.path.lower().endswith('.docx')]

## Save

In [0]:
for file in files_list_txt:

    # Load configuration
    file_name = file.name
    print(file_name)

    format_file, is_visually_checked, is_business, is_rejection, non_updated_type = file_parameters(file_name, config)

    # Define file paths for the format XML file and content TXT files
    xml_file_path = f"/Volumes/p-cdh-delivery/tiresias/rejection_files/formats/{format_file}"
    txt_file_path = f'/Volumes/p-cdh-delivery/tiresias/rejection_files/extracted_rejection_files/{file_name}'

    parsed_df = parse_rejections_files_format(xml_file_path, txt_file_path)

    # in case of REJOPT in the filename
    if "REJOPT" in txt_file_path:
        # Collect the rows as a list
        rows = parsed_df.collect()
        # Remove the last row
        rows_without_last = rows[:-1]
        # Create a new DataFrame from the remaining rows
        parsed_df = spark.createDataFrame(rows_without_last, parsed_df.columns)

    parsed_df = parsed_df.withColumn("is_visually_checked", F.lit(is_visually_checked)) \
                .withColumn("is_business", F.lit(is_business)) \
                .withColumn("is_rejection", F.lit(is_rejection)) \
                .withColumn("non_updated_type", F.lit(non_updated_type)) \
                .withColumn("Filename", F.lit(file_name)) \
                .withColumn("ExportDate", F.lit(export_date))


    #insert data in 
    export_type = 'B' if is_business == 1 else 'C'

    if is_rejection == 1:
        if is_visually_checked == 1:
            df_filtered =  parsed_df.select(
                lit(export_type).alias("export_type"),
                lit(export_date).alias("export_date"),
                lit(from_utc_timestamp(current_timestamp(), "Europe/Athens")).alias("IngestionTimestamp"),
                lit(file_name).alias("rejection_file"),
                lit(None).alias("vendor_code"),
                col("LoanCardCode").alias("account_number"),
                col("VATNumber").alias("Customer_1"),
                col("Identification").alias("Customer_2"),
                expr("LEFT(Reasoning, 100)").alias("error_code"),
                lit(None).alias("cl_sk_id")
            )

            print('is visually checked & is r ejection')
            df_filtered.write.format("delta").mode("append").saveAsTable(f"`{env}-cdh-delivery`.tiresias.old_style_rejections")

        else:
            rejections_list = []
            for index in range(1, 4):
                temp_df = parsed_df.filter(col(f"Reason{index}").isNotNull()).select(
                    lit(export_type).alias("export_type"),
                    lit(export_date).alias("export_date"),
                    lit(from_utc_timestamp(current_timestamp(), "Europe/Athens")).alias("IngestionTimestamp"),
                    lit(file_name).alias("rejection_file"),
                    lit(None).alias("vendor_code"),
                    col("AccountCode").alias("account_number"),
                    col("VATNumber").alias("Customer_1"),
                    col("Identification").alias("Customer_2"),
                    expr(f"LEFT(Reason{index}, 100)").alias("error_code"),
                    lit(None).alias("cl_sk_id")
                )
                rejections_list.append(temp_df)
            df_rejections = rejections_list[0]
            for temp_df in rejections_list[1:]:
                df_rejections = df_rejections.union(temp_df)

            df_rejections.write.format("delta").mode("append").saveAsTable(f"`{env}-cdh-delivery`.tiresias.old_style_rejections")
